In [4]:
!pip install pandas scikit-learn
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

#step 1:Load CSV
data = pd.read_csv("plagiarism_dataset.csv")
print("Dataset loaded successfully")
print("Number of documents:",len(data))

#step 2: Clean and normalize text
cleaned_text  = []
for text in data["text"]:
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]"," ",text)
    text = re.sub(r"\s+"," ",text)
    cleaned_text.append(text)

#step 3: TF-IDF Vectorization
vectorizer = TfidfVectorizer(stop_words="english")
tfidf = vectorizer.fit_transform(cleaned_text)
print("TF-IDF Conversion completed")

#step 4:Calculate cosine similarity
similarity = cosine_similarity(tfidf)

#step5: display similarity matrix
matrix = pd.DataFrame(similarity*100,index = data["document_id"],columns=data["document_id"])
print("\nSimilarity Matrix:")
print(matrix.round(2))

#step 6: Set plagiarism threshold
threshold = 70

#step 7: find document pairs
results = []

for i in range(len(data)):
    for j in range(i+1,len(data)):
        score = similarity[i][j]*100
        if score >= threshold:
            result = "Possible Plagiarism"
        else:
            result = "Low Similarity"
        results.append([data["document_id"].iloc[i],data["document_id"].iloc[j],round(score,2),result])

#step 8: Create similarity report
report = pd.DataFrame(results,columns=["Document 1","Document 2","Similarity %","Result"])

#step 9: Rank by similarity
report = report.sort_values(by="Similarity %",ascending=False)
print("\nRanked Similarity Report:")
print(report.to_string(index=False))

#step 10: Save report
report.to_csv("similarity_report.csv",index=False)
print("\nSimilarity report saved as similarity_report.csv")


Dataset loaded successfully
Number of documents: 10
TF-IDF Conversion completed

Similarity Matrix:
document_id       A       B       C       D       E       F       G       H  \
document_id                                                                   
A            100.00   82.83    0.00   22.84   32.67    8.24   29.56    0.00   
B             82.83  100.00    0.00   19.24   29.39    8.34   29.92    0.00   
C              0.00    0.00  100.00    0.00    0.00    0.00    0.00   13.54   
D             22.84   19.24    0.00  100.00   12.46    0.00    4.39    1.82   
E             32.67   29.39    0.00   12.46  100.00    4.86   26.82    0.00   
F              8.24    8.34    0.00    0.00    4.86  100.00    5.55    0.00   
G             29.56   29.92    0.00    4.39   26.82    5.55  100.00    0.00   
H              0.00    0.00   13.54    1.82    0.00    0.00    0.00  100.00   
I              9.08    9.19    6.41   36.25    5.98    1.47    6.12    1.70   
J              3.32    3.36    